In [1]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime
print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import TOKENIZER_SAVE_DIRECTORY, OUR_DATASET_DIRECTORY
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model, Verbose
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

added '/home/holo/dev/PFE_LLM_art_generation' to import paths
sys.version_info(major=3, minor=10, micro=19, releaselevel='final', serial=0)


In [2]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [3]:
import importlib
import LLM.model
_ = importlib.reload(LLM.model)
from LLM.model import Model

In [ ]:
try:
    torch.cuda.empty_cache()
    del model # type: ignore
    torch.cuda.empty_cache()
except Exception: pass
model = Model(
    save_name="tmp-tests", tokenizer=TOKENIZER_SAVE_DIRECTORY.joinpath("our_tokenizer"), 
    device="cuda", depth=6, head_dim=128, context_size=4096, nb_heads_mult=1.0)
model.show_infos()

loading the tokenizer from: /home/holo/dev/PFE_LLM_art_generation/models_save/tmp-tests/tokenizer.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(128/768) = 2.449490
GPTConfig(sequence_len=8192, vocab_size=585, n_layer=6, n_head=1, n_kv_head=1, n_embd=128, window_pattern='SSSL')
1_589_356 total params (embeding: 327_680 | last layer: 81_920 | transformer: 1_179_744)
on device: device(type='cuda', index=0), with effective context_size: 4096


In [7]:
model.save("un_trained")

saving the tokenizer to: /home/holo/dev/PFE_LLM_art_generation/models_save/tmp-tests/tokenizer.json
saving the historique to: /home/holo/dev/PFE_LLM_art_generation/models_save/tmp-tests/versions/version_0004_un_trained/history.json


(4,
 PosixPath('/home/holo/dev/PFE_LLM_art_generation/models_save/tmp-tests/versions/version_0004_un_trained'))

In [6]:
try:
    if "dataset" in globals() : raise FileExistsError
    dataset = svg_dataset.SVGDataset(
        OUR_DATASET_DIRECTORY, context_size=model.context_size,
        tokenizer=model.tokenizer.encode, decoder=model.tokenizer.decode)
except FileExistsError: pass

In [7]:
model.train(
    dataset=dataset, batch_size=16, 
    nbEpoches=1, timeLimite=999_999, verbose=Verbose.liveProgress)


starting epoch: 1
finished batches in 2 min 5.6 sec                
finished epoches in 2 min 5.6 sec
trained on: 863 batch (13793 chuncks) in 1 min 54.0 sec
 -> 7.57 batch/sec | 121.03 chuncks/sec
 -> CE: 1.026 | PPL: 2.789 | top-1: 74.69%


In [8]:
model.save("trained-1epoche")

saving the tokenizer to: /home/holo/dev/PFE_LLM_art_generation/models_save/tmp-tests/tokenizer.json
saving the historique to: /home/holo/dev/PFE_LLM_art_generation/models_save/tmp-tests/versions/version_0001_trained-1epoche/history.json


(1,
 PosixPath('/home/holo/dev/PFE_LLM_art_generation/models_save/tmp-tests/versions/version_0001_trained-1epoche'))

In [7]:
model.train(
    dataset=dataset, batch_size=16, 
    nbEpoches=4, timeLimite=999_999, verbose=Verbose.liveProgress)

starting epoch: 1
finished batches in 2 min 1.5 sec               
doing epoches: 25.00 % done, rem: 6 min 4.5 sec
trained on: 863 batch (13793 chuncks) in 1 min 49.8 sec
 -> 7.86 batch/sec | 125.58 chuncks/sec
 -> CE: 0.7707 | PPL: 2.161 | top-1: 81.10%
starting epoch: 2
finished batches in 1 min 57.7 sec              
doing epoches: 50.00 % done, rem: 3 min 55.5 sec
trained on: 863 batch (13793 chuncks) in 1 min 49.0 sec
 -> 7.92 batch/sec | 126.59 chuncks/sec
 -> CE: 0.757 | PPL: 2.132 | top-1: 81.49%
starting epoch: 3
finished batches in 2 min 0.7 sec               
doing epoches: 75.00 % done, rem: 2 min 0.7 sec 
trained on: 863 batch (13793 chuncks) in 1 min 49.0 sec
 -> 7.92 batch/sec | 126.53 chuncks/sec
 -> CE: 0.7437 | PPL: 2.104 | top-1: 81.73%
starting epoch: 4
finished batches in 1 min 57.5 sec              
finished epoches in 7 min 57.4 sec             
trained on: 863 batch (13793 chuncks) in 1 min 48.8 sec
 -> 7.93 batch/sec | 126.73 chuncks/sec
 -> CE: 0.7346 | PPL: 2